# Pet Adoption Classification - part 3, optimization

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import LinearSegmentedColormap
import time, warnings
from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold, GridSearchCV, cross_val_score
from sklearn.feature_selection import SelectKBest, f_classif, mutual_info_classif
from sklearn.base import clone
from pyswarm import pso
from flaml import AutoML
import optuna
import shap

warnings.filterwarnings('ignore')

colors = ['#9b5de5', '#f15bb5', '#fee440', '#00bbf9', '#00f5d4']
cmap_custom = LinearSegmentedColormap.from_list('custom', ['#ffffff', '#9b5de5'])

RANDOM_STATE = 42
TARGET = 'AdoptionLikelihood'
sns.set_style('whitegrid')

## Helper functions copied from part 2

In [ ]:
def manual_train_test_split(df, target_column, test_size=0.2, random_state=42):
    random_generator = np.random.default_rng(random_state)

    train_idx = []
    test_idx = []

    for class_label in df[target_column].unique():
        class_idx = df.index[df[target_column] == class_label].tolist()

        random_generator.shuffle(class_idx)

        num_test_samples = max(1, int(len(class_idx) * test_size))

        test_idx.extend(class_idx[:num_test_samples])
        train_idx.extend(class_idx[num_test_samples:])

    train_set = df.loc[train_idx].sample(frac=1, random_state=random_state).reset_index(drop=True)
    test_set = df.loc[test_idx].sample(frac=1, random_state=random_state).reset_index(drop=True)

    return train_set, test_set


def manual_stratified_kfold(df, target_column, n_splits=5, random_state=42):
    '''Generator yielding stratified (train_idx, val_idx) splits as positional indices.'''
    rng = np.random.default_rng(random_state)

    folds_per_class = {}
    for class_label in sorted(df[target_column].unique()):
        positions = np.where(df[target_column].to_numpy() == class_label)[0]
        rng.shuffle(positions)
        # Distribute this class's samples evenly into n_splits folds
        folds_per_class[class_label] = np.array_split(positions, n_splits)

    for fold_i in range(n_splits):
        val_idx = np.concatenate([folds_per_class[c][fold_i] for c in folds_per_class])
        train_idx = np.concatenate([
            folds_per_class[c][j]
            for c in folds_per_class
            for j in range(n_splits) if j != fold_i
        ])

        rng.shuffle(train_idx)
        rng.shuffle(val_idx)

        yield train_idx, val_idx

In [ ]:
def manual_confusion_matrix(y_true, y_pred, num_classes=2):
    y_true = np.asarray(y_true).astype(int)
    y_pred = np.asarray(y_pred).astype(int)
    cm = np.zeros((num_classes, num_classes), dtype=int)
    for true_label, pred_label in zip(y_true, y_pred):
        cm[true_label, pred_label] += 1
    return cm


def manual_accuracy(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    return float(np.mean(y_true == y_pred))


def manual_precision(y_true, y_pred, positive_class=1):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    tp = int(np.sum((y_pred == positive_class) & (y_true == positive_class)))
    fp = int(np.sum((y_pred == positive_class) & (y_true != positive_class)))
    return tp / (tp + fp) if (tp + fp) > 0 else 0.0


def manual_recall(y_true, y_pred, positive_class=1):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    tp = int(np.sum((y_pred == positive_class) & (y_true == positive_class)))
    fn = int(np.sum((y_pred != positive_class) & (y_true == positive_class)))
    return tp / (tp + fn) if (tp + fn) > 0 else 0.0


def manual_f1(y_true, y_pred, positive_class=1):
    p = manual_precision(y_true, y_pred, positive_class)
    r = manual_recall(y_true, y_pred, positive_class)
    return 2 * p * r / (p + r) if (p + r) > 0 else 0.0


def manual_roc_auc(y_true, y_score):
    '''Compute ROC-AUC via trapezoidal integration of the ROC curve.'''
    y_true = np.asarray(y_true)
    y_score = np.asarray(y_score)

    # Sort samples by descending score
    order = np.argsort(-y_score, kind='mergesort')
    y_true_sorted = y_true[order]

    P = int(np.sum(y_true == 1))
    N = int(np.sum(y_true == 0))
    if P == 0 or N == 0:
        return 0.5

    # TPR = cumulative positives / total positives
    # FPR = cumulative negatives / total negatives
    tpr = np.cumsum(y_true_sorted == 1) / P
    fpr = np.cumsum(y_true_sorted == 0) / N

    # Prepend (0, 0) so curve starts at origin
    tpr = np.concatenate(([0.0], tpr))
    fpr = np.concatenate(([0.0], fpr))

    return float(np.trapz(tpr, fpr))


def evaluate_cv(model, train_df, feature_cols, target_col, n_splits=5, random_state=RANDOM_STATE):
    """Run stratified k-fold CV and return per-fold metric arrays."""
    fold_metrics = {'accuracy': [], 'precision': [], 'recall': [], 'f1': [], 'roc_auc': [], 'fit_time': []}

    for tr_idx, vl_idx in manual_stratified_kfold(train_df, target_col, n_splits=n_splits, random_state=random_state):
        X_tr = train_df.iloc[tr_idx][feature_cols].to_numpy()
        y_tr = train_df.iloc[tr_idx][target_col].to_numpy()
        X_vl = train_df.iloc[vl_idx][feature_cols].to_numpy()
        y_vl = train_df.iloc[vl_idx][target_col].to_numpy()

        # Clone-via-class-reinit is unnecessary since we instantiate fresh models per dataset.
        t0 = time.time()
        model.fit(X_tr, y_tr)
        fit_time = time.time() - t0

        y_pred = model.predict(X_vl)
        y_proba = model.predict_proba(X_vl)[:, 1]

        fold_metrics['accuracy'].append(manual_accuracy(y_vl, y_pred))
        fold_metrics['precision'].append(manual_precision(y_vl, y_pred))
        fold_metrics['recall'].append(manual_recall(y_vl, y_pred))
        fold_metrics['f1'].append(manual_f1(y_vl, y_pred))
        fold_metrics['roc_auc'].append(manual_roc_auc(y_vl, y_proba))
        fold_metrics['fit_time'].append(fit_time)

    return {k: np.array(v) for k, v in fold_metrics.items()}


def print_cv_summary(cv, label=""):
    print(f"{'':=<55}")
    if label: print(f'  {label}')
    for m in ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']:
        print(f"  {m:<12} {cv[m].mean():.4f} ± {cv[m].std():.4f}")
    print(f"  fit_time   {cv['fit_time'].mean():.3f}s per fold")


def cv_summary_row(cv, label):
    """Return a one-row DataFrame summary of a CV result."""
    return pd.DataFrame([{
        'method': label,
        'accuracy': cv['accuracy'].mean(),
        'precision': cv['precision'].mean(),
        'recall': cv['recall'].mean(),
        'f1': cv['f1'].mean(),
        'roc_auc': cv['roc_auc'].mean(),
        'fit_time': cv['fit_time'].mean(),
        'f1_std': cv['f1'].std(),
    }])

## Baseline model

In [ ]:
df = pd.read_csv('data_after_processing/df_knn_std.csv')

train_df, test_df = manual_train_test_split(df, TARGET, test_size=0.2, random_state=RANDOM_STATE)
feature_cols = [c for c in df.columns if c != TARGET]

baseline_model = CatBoostClassifier(iterations=200, random_seed=RANDOM_STATE, verbose=0)

baseline_cv = evaluate_cv(baseline_model, train_df, feature_cols, TARGET)

baseline_row = {
    'dataset': 'df_knn_std',
    'model': 'CatBoost',
    'accuracy_mean': baseline_cv['accuracy'].mean(),
    'precision_mean': baseline_cv['precision'].mean(),
    'recall_mean': baseline_cv['recall'].mean(),
    'f1_mean': baseline_cv['f1'].mean(),
    'roc_auc_mean': baseline_cv['roc_auc'].mean(),
    'fit_time_mean': baseline_cv['fit_time'].mean()
}

df_baseline = pd.DataFrame([baseline_row])

cols_to_round = [
    'accuracy_mean',
    'precision_mean',
    'recall_mean',
    'f1_mean',
    'roc_auc_mean',
    'fit_time_mean'
]

df_baseline[cols_to_round] = df_baseline[cols_to_round].round(4)

df_baseline

In [ ]:
results_table = cv_summary_row(baseline_cv, 'Baseline (default CatBoost, all features)')

## Feature optimization

### Correlation Matrix

In [ ]:
corr_matrix = train_df[feature_cols].corr(method='pearson')

In [ ]:
fig, ax = plt.subplots(figsize=(12, 10))

mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

sns.heatmap(
    corr_matrix,
    mask=mask,
    annot=True,
    annot_kws={'size': 8},
    fmt='.2f',
    cmap='Purples',
    vmin=-1,
    vmax=1,
    center=0,
    ax=ax,
    linewidths=0.5,
    cbar_kws={'shrink': 0.8}
)

ax.set_title('Macierz korelacji Pearsona', fontsize=16)

plt.show()

In [ ]:
threshold = 0.9
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

to_drop = [column for column in upper.columns if any(upper[column] > threshold)]

print('To be deleted:', to_drop)

In [ ]:
for i in range(len(corr_matrix.columns)):
    for j in range(i):
        if abs(corr_matrix.iloc[i, j]) > 0.9:
            col_i = corr_matrix.columns[i]
            col_j = corr_matrix.columns[j]

            var_i = train_df[col_i].var()
            var_j = train_df[col_j].var()

            drop_candidate = col_i if var_i < var_j else col_j
            keep_candidate = col_j if var_i < var_j else col_i

            print(f"Corr({col_i}, {col_j}) = {corr_matrix.iloc[i, j]:.3f} "
                  f"drop '{drop_candidate}' (var={train_df[drop_candidate].var():.4f}), "
                  f"keep '{keep_candidate}' (var={train_df[keep_candidate].var():.4f})")

In [ ]:
# Dodatkowo możemy zostawić cechę z większą wariancją
to_drop = set()

for i in range(len(corr_matrix.columns)):
    for j in range(i):
        if abs(corr_matrix.iloc[i, j]) > 0.9:

            col_i = corr_matrix.columns[i]
            col_j = corr_matrix.columns[j]

            # usuń tę z mniejszą wariancją
            if train_df[col_i].var() < train_df[col_j].var():
                to_drop.add(col_i)
            else:
                to_drop.add(col_j)

In [ ]:
CORR_THRESHOLD = 0.90

upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
to_drop_corr = [col for col in upper.columns if any(upper[col].abs() > CORR_THRESHOLD)]
print(f'Features to drop (correlation > {CORR_THRESHOLD}): {to_drop_corr}')

feature_cols_corr = [f for f in feature_cols if f not in to_drop_corr]
print(f'Feature count: {len(feature_cols)}  {len(feature_cols_corr)}')

In [ ]:
print_cv_summary(baseline_cv, f'Before Feature Optimization')

In [ ]:
model_corr = clone(baseline_model)
cv_corr = evaluate_cv(model_corr, train_df, feature_cols_corr, TARGET)
print_cv_summary(cv_corr, f'After correlation filter ({len(feature_cols_corr)} features)')


### SelectKBest - rank features by F-test and mutual information

In [ ]:
X = train_df[feature_cols_corr].values
y = train_df[TARGET].values

# F-test
selector_f = SelectKBest(f_classif, k='all').fit(X, y)
scores_f = pd.DataFrame(
    {'f_score': selector_f.scores_},
    index=feature_cols_corr
).sort_values(by='f_score', ascending=False)

selector_mi = SelectKBest(mutual_info_classif, k='all').fit(X, y)
scores_mi = pd.DataFrame(
    {'mi_score': selector_mi.scores_},
    index=feature_cols_corr
).sort_values(by='mi_score', ascending=False)

display(scores_f.head(20))
display(scores_mi.head(20))

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
scores_f.head(15).plot(kind='barh', ax=ax1, color='#9b5de5')
ax1.set_title('SelectKBest F-test (ANOVA)')
ax1.invert_yaxis()

scores_mi.head(15).plot(kind='barh', ax=ax2, color='#00bbf9')
ax2.set_title('SelectKBest Mutual Information')
ax2.invert_yaxis()

plt.tight_layout()
plt.show()

In [ ]:
top_f = set(scores_f.head(15).index)
top_mi = set(scores_mi.head(15).index)

features_intersection = list(top_f & top_mi)

print('Intersection features:')
print(features_intersection)

In [ ]:
# Sweep K and pick the best for each scoring method
K_values = [5, 8, 10, 12, 15, 20]
sweep_rows = []

for K in K_values:
    feats_f = scores_f.head(K).index.tolist()
    feats_mi = scores_mi.head(K).index.tolist()
    cv_kf = evaluate_cv(clone(baseline_model), train_df, feats_f, TARGET)
    cv_kmi = evaluate_cv(clone(baseline_model), train_df, feats_mi, TARGET)
    sweep_rows.append({'K': K, 'method': 'F-test', 'f1': cv_kf['f1'].mean(), 'roc_auc': cv_kf['roc_auc'].mean()})
    sweep_rows.append(
        {'K': K, 'method': 'Mutual Information', 'f1': cv_kmi['f1'].mean(), 'roc_auc': cv_kmi['roc_auc'].mean()})

sweep_df = pd.DataFrame(sweep_rows)
display(sweep_df.pivot(index='K', columns='method', values='f1').round(4).rename_axis(columns='F1 (CV mean)'))

In [ ]:
# Plot the K-sweep
fig, ax = plt.subplots(figsize=(9, 5))
for method, group in sweep_df.groupby('method'):
    ax.plot(group['K'], group['f1'], marker='o', label=method, linewidth=2)
ax.set_xlabel('K (number of features)')
ax.set_ylabel('F1 (CV mean)')
ax.set_title('SelectKBest - F1 vs K')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# Pick best K and method
best_sweep = sweep_df.sort_values('f1', ascending=False).iloc[0]
K_best, method_best = int(best_sweep['K']), best_sweep['method']
print(f"Best SelectKBest config: method={method_best}, K={K_best}, F1={best_sweep['f1']:.4f}")

scores_best = scores_f if method_best == 'F-test' else scores_mi
features_kbest = scores_best.head(K_best).index.tolist()

In [ ]:
cv_kbest = evaluate_cv(clone(baseline_model), train_df, features_kbest, TARGET)
print_cv_summary(cv_kbest, f'SelectKBest - {method_best}, K={K_best}')

results_table = pd.concat([
    results_table,
    cv_summary_row(cv_kbest, f'SelectKBest ({method_best}, K={K_best})')
], ignore_index=True)

In [ ]:
final_features = features_kbest
print('Final features:')
print(final_features)

### Feature-selection comparison

In [ ]:
# Compare the three feature-selection methods on identical baseline CatBoost
feature_sel_summary = pd.DataFrame([
    {'method': 'Baseline (all features)', 'n_features': len(feature_cols), 'f1': baseline_cv['f1'].mean(),
     'roc_auc': baseline_cv['roc_auc'].mean()},
    {'method': 'Correlation filter', 'n_features': len(feature_cols_corr), 'f1': cv_corr['f1'].mean(),
     'roc_auc': cv_corr['roc_auc'].mean()},
    {'method': f'SelectKBest ({method_best}, K={K_best})', 'n_features': len(features_kbest),
     'f1': cv_kbest['f1'].mean(), 'roc_auc': cv_kbest['roc_auc'].mean()},
]).round(4)

display(feature_sel_summary)

# Pick winner by F1
best_feature_row = feature_sel_summary.iloc[feature_sel_summary['f1'].idxmax()]
print(f"Winning feature-selection method: {best_feature_row['method']}  (F1={best_feature_row['f1']:.4f})")

# Map back to actual list
if 'Baseline' in best_feature_row['method']:
    final_features = feature_cols
elif 'Correlation' in best_feature_row['method']:
    final_features = feature_cols_corr
elif 'SelectKBest' in best_feature_row['method']:
    final_features = features_kbest

print(f'Final feature set ({len(final_features)} features): {final_features}')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
methods = feature_sel_summary['method'].tolist()
f1s = feature_sel_summary['f1'].tolist()
bars = ax.barh(methods, f1s, color=colors[:len(methods)])
for bar, val in zip(bars, f1s):
    ax.text(val + 0.001, bar.get_y() + bar.get_height() / 2, f'{val:.4f}', va='center', fontsize=10)
ax.set_xlabel('F1 (CV mean)')
ax.set_title('Feature-selection method comparison')
ax.axvline(baseline_cv['f1'].mean(), color='gray', linestyle='--', alpha=0.6, label='Baseline F1')
ax.legend()
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

 ## Hyperparameter optimization

In [ ]:
# Prepare data with the chosen feature set
X_train = train_df[final_features].values
y_train = train_df[TARGET].values
X_test = test_df[final_features].values
y_test = test_df[TARGET].values
cv_splitter = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

print(f'X_train shape: {X_train.shape}')
print(f'X_test shape:  {X_test.shape}')

### GridSearchCV

In [ ]:
grid_param_space = {
    'iterations': [200, 400],
    'depth': [4, 6, 8],
    'learning_rate': [0.03, 0.1],
    'l2_leaf_reg': [1, 3, 7],
}
print(f'Grid size: {2 * 3 * 2 * 3} = 36 combinations × 5 folds = 180 fits')

t0 = time.time()
grid = GridSearchCV(
    estimator=CatBoostClassifier(random_seed=RANDOM_STATE, verbose=0),
    param_grid=grid_param_space,
    scoring='f1',
    cv=cv_splitter,
    n_jobs=-1,
    refit=True,
    return_train_score=False,
)
grid.fit(X_train, y_train)
grid_time = time.time() - t0

print(f'GridSearch elapsed: {grid_time:.1f}s')
print(f'Best params: {grid.best_params_}')
print(f'Best CV F1:  {grid.best_score_:.4f}')

In [ ]:
grid_best_model = CatBoostClassifier(**grid.best_params_, random_seed=RANDOM_STATE, verbose=0)
cv_grid = evaluate_cv(grid_best_model, train_df, final_features, TARGET)
print_cv_summary(cv_grid, 'After GridSearchCV')

results_table = pd.concat([
    results_table,
    cv_summary_row(cv_grid, 'GridSearchCV (tuned CatBoost)')
], ignore_index=True)

In [ ]:
grid_cv_df = pd.DataFrame(grid.cv_results_)
top5_grid = grid_cv_df.nlargest(5, 'mean_test_score')[['params', 'mean_test_score', 'std_test_score']].reset_index(
    drop=True)
top5_grid['mean_test_score'] = top5_grid['mean_test_score'].round(4)
top5_grid['std_test_score'] = top5_grid['std_test_score'].round(4)
display(top5_grid)

### Optuna

In [ ]:
optuna.logging.set_verbosity(optuna.logging.WARNING)


def objective_optuna(trial):
    params = {
        'iterations': trial.suggest_int('iterations', 150, 600),
        'depth': trial.suggest_int('depth', 3, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 0.5, 10.0, log=True),
        'random_seed': RANDOM_STATE,
        'verbose': 0,
    }
    model = CatBoostClassifier(**params)
    scores = cross_val_score(model, X_train, y_train, scoring='f1', cv=cv_splitter, n_jobs=-1)
    return scores.mean()


t0 = time.time()
study = optuna.create_study(
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE),
)
study.optimize(objective_optuna, n_trials=40, show_progress_bar=False)
optuna_time = time.time() - t0

print(f'Optuna elapsed: {optuna_time:.1f}s')
print(f'Best params: {study.best_params}')
print(f'Best CV F1:  {study.best_value:.4f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

trial_vals = [t.value for t in study.trials if t.value is not None]
running_best = np.maximum.accumulate(trial_vals)
axes[0].plot(trial_vals, marker='o', linewidth=1, alpha=0.6, label='Trial F1', color='#9b5de5')
axes[0].plot(running_best, color='#00bbf9', linewidth=2, label='Running best')
axes[0].axhline(grid.best_score_, color='gray', linestyle='--', alpha=0.6,
                label=f'GridSearch best ({grid.best_score_:.4f})')
axes[0].set_xlabel('Trial')
axes[0].set_ylabel('F1 (CV mean)')
axes[0].set_title('Optuna optimization history')
axes[0].legend()
axes[0].grid(alpha=0.3)

try:
    importances = optuna.importance.get_param_importances(study)
    axes[1].barh(list(importances.keys()), list(importances.values()), color='#f15bb5')
    axes[1].set_title('Optuna - hyperparameter importance')
    axes[1].invert_yaxis()
except Exception as e:
    axes[1].text(0.5, 0.5, f'importance unavailable:\n{e}', ha='center', va='center')

plt.tight_layout()
plt.show()

In [ ]:
optuna_best_model = CatBoostClassifier(**study.best_params, random_seed=RANDOM_STATE, verbose=0)
cv_optuna = evaluate_cv(optuna_best_model, train_df, final_features, TARGET)
print_cv_summary(cv_optuna, 'After Optuna')

results_table = pd.concat([
    results_table,
    cv_summary_row(cv_optuna, 'Optuna (tuned CatBoost)')
], ignore_index=True)

 ### Particle Swarm Optimization (PSO)

In [ ]:
pso_lb = [150, 3, np.log(0.01), np.log(0.5)]
pso_ub = [600, 10, np.log(0.3), np.log(10.0)]

pso_history = []

def objective_pso(x):
    params = {
        'iterations': int(round(x[0])),
        'depth': int(round(x[1])),
        'learning_rate': float(np.exp(x[2])),
        'l2_leaf_reg': float(np.exp(x[3])),
        'random_seed': RANDOM_STATE,
        'verbose': 0,
    }
    model = CatBoostClassifier(**params)
    f1 = cross_val_score(model, X_train, y_train, scoring='f1', cv=cv_splitter, n_jobs=-1).mean()
    pso_history.append(f1)
    return -f1  # pyswarm MINIMIZES, so negate


t0 = time.time()
pso_result = pso(
    objective_pso, pso_lb, pso_ub,
    swarmsize=15,
    maxiter=10,
    omega=0.5,
    phip=0.5,
    phig=0.5,
    minstep=1e-6,
    minfunc=1e-6,
    debug=False,
)


In [ ]:
pso_best_x = pso_result['x']
pso_best_neg = pso_result['fun']
pso_time = time.time() - t0

pso_best_params = {
    'iterations': int(round(pso_best_x[0])),
    'depth': int(round(pso_best_x[1])),
    'learning_rate': float(np.exp(pso_best_x[2])),
    'l2_leaf_reg': float(np.exp(pso_best_x[3])),
}
pso_best_score = -pso_best_neg

print(f'PSO elapsed: {pso_time:.1f}s')
print(f'Total evaluations: {len(pso_history)}')
print(f'Best params: {pso_best_params}')
print(f'Best CV F1:  {pso_best_score:.4f}')

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
trial_idx = np.arange(1, len(pso_history) + 1)
running_best = np.maximum.accumulate(pso_history)

ax.plot(trial_idx, pso_history, marker='.', linewidth=1, alpha=0.5, label='Per-evaluation F1', color='#9b5de5')
ax.plot(trial_idx, running_best, color='#00bbf9', linewidth=2, label='Running best')
ax.axhline(grid.best_score_, color='gray', linestyle='--', alpha=0.6, label=f'GridSearch best ({grid.best_score_:.4f})')
ax.axhline(study.best_value, color='orange', linestyle='--', alpha=0.6, label=f'Optuna best ({study.best_value:.4f})')
ax.set_xlabel('Evaluation number')
ax.set_ylabel('F1 (CV mean)')
ax.set_title('PSO - convergence')
ax.legend(loc='lower right')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Evaluate PSO's best on our manual CV pipeline (consistent metric set)
pso_best_model = CatBoostClassifier(**pso_best_params, random_seed=RANDOM_STATE, verbose=0)
cv_pso = evaluate_cv(pso_best_model, train_df, final_features, TARGET)
print_cv_summary(cv_pso, 'After PSO (pyswarm)')

results_table = pd.concat([
    results_table,
    cv_summary_row(cv_pso, 'PSO / pyswarm (tuned CatBoost)')
], ignore_index=True)

## flaML

In [ ]:
automl = AutoML()
automl_settings = {
    'time_budget': 90,
    'metric': 'f1',
    'task': 'classification',
    'eval_method': 'cv',
    'n_splits': 5,
    'seed': RANDOM_STATE,
    'verbose': 0,
    'estimator_list': ['lgbm', 'xgboost', 'catboost', 'rf', 'extra_tree', 'lrl1'],
}

t0 = time.time()
automl.fit(X_train=X_train, y_train=y_train, **automl_settings)
flaml_time = time.time() - t0

print(f'FLAML elapsed: {flaml_time:.1f}s')
print(f'Best estimator: {automl.best_estimator}')
print(f'Best hyperparams: {automl.best_config}')
print(f'Best CV F1 (loss = 1 - F1): {1 - automl.best_loss:.4f}')

In [ ]:
flaml_best_model = automl.model.estimator
cv_flaml = evaluate_cv(flaml_best_model, train_df, final_features, TARGET)
print_cv_summary(cv_flaml, f'FLAML AutoML (best: {automl.best_estimator})')

results_table = pd.concat([
    results_table,
    cv_summary_row(cv_flaml, f'FLAML AutoML (best: {automl.best_estimator})')
], ignore_index=True)

In [ ]:
history = pd.DataFrame(automl.config_history.values()) if hasattr(automl, 'config_history') else None
if history is not None and len(history) > 0:
    display(history.head(10))
else:
    print(f'FLAML przetestował {len(automl.classes_)} klas, najlepszy: {automl.best_estimator}')

## Final comparison of all methods

In [ ]:
display(results_table.round(4))

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))
order = results_table.sort_values('f1')
bar_colors = ['#cccccc'] + [colors[i % len(colors)] for i in range(len(order) - 1)]
bars = ax.barh(order['method'], order['f1'], color=bar_colors)
baseline_f1 = results_table.iloc[0]['f1']
ax.axvline(baseline_f1, color='gray', linestyle='--', alpha=0.7, label=f'Baseline F1 ({baseline_f1:.4f})')
for bar, val in zip(bars, order['f1']):
    ax.text(val + 0.0005, bar.get_y() + bar.get_height() / 2, f'{val:.4f}', va='center', fontsize=9)
ax.set_xlabel('F1 (CV mean)')
ax.set_title('All methods - CV F1 comparison')
ax.legend()
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
gain = results_table.copy()
gain['Δ_f1_vs_baseline'] = (gain['f1'] - baseline_cv['f1'].mean()).round(4)
gain['Δ_roc_auc_vs_baseline'] = (gain['roc_auc'] - baseline_cv['roc_auc'].mean()).round(4)
display(gain[['method', 'f1', 'Δ_f1_vs_baseline', 'roc_auc', 'Δ_roc_auc_vs_baseline']].round(4))

## Final evaluation on the test set

In [ ]:
def test_eval(model, X_tr, y_tr, X_te, y_te, label):
    m = clone(model)
    t0 = time.time()
    m.fit(X_tr, y_tr)
    fit_time = time.time() - t0
    y_pred = m.predict(X_te)
    y_proba = m.predict_proba(X_te)[:, 1]
    return {
        'method': label,
        'accuracy': manual_accuracy(y_te, y_pred),
        'precision': manual_precision(y_te, y_pred),
        'recall': manual_recall(y_te, y_pred),
        'f1': manual_f1(y_te, y_pred),
        'roc_auc': manual_roc_auc(y_te, y_proba),
        'fit_time': fit_time,
    }, m, y_pred, y_proba


X_train_full = train_df[feature_cols].values
X_test_full = test_df[feature_cols].values

test_rows = []
fitted = {}

row, m_base, _, _ = test_eval(
    CatBoostClassifier(iterations=200, random_seed=RANDOM_STATE, verbose=0),
    X_train_full, y_train, X_test_full, y_test, 'Baseline (default, all feats)'
)
test_rows.append(row)
fitted['Baseline'] = m_base

row, m_grid, _, _ = test_eval(
    CatBoostClassifier(**grid.best_params_, random_seed=RANDOM_STATE, verbose=0),
    X_train, y_train, X_test, y_test, 'GridSearchCV (tuned)'
)
test_rows.append(row)
fitted['Grid'] = m_grid

row, m_opt, _, _ = test_eval(
    CatBoostClassifier(**study.best_params, random_seed=RANDOM_STATE, verbose=0),
    X_train, y_train, X_test, y_test, 'Optuna (tuned)'
)
test_rows.append(row)
fitted['Optuna'] = m_opt

row, m_pso, _, _ = test_eval(
    CatBoostClassifier(**pso_best_params, random_seed=RANDOM_STATE, verbose=0),
    X_train, y_train, X_test, y_test, 'PSO / pyswarm (tuned)'
)
test_rows.append(row)
fitted['PSO'] = m_pso

test_results = pd.DataFrame(test_rows).round(4)
display(test_results)

In [ ]:
compare_cv_test = pd.DataFrame({
    'method': ['Baseline', 'GridSearchCV', 'Optuna', 'PSO (pyswarm)'],
    'CV F1': [baseline_cv['f1'].mean(), cv_grid['f1'].mean(), cv_optuna['f1'].mean(), cv_pso['f1'].mean()],
    'Test F1': test_results['f1'].tolist(),
}).round(4)
compare_cv_test['CV → Test drop'] = (compare_cv_test['CV F1'] - compare_cv_test['Test F1']).round(4)
display(compare_cv_test)

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(compare_cv_test))
w = 0.35
ax.bar(x - w / 2, compare_cv_test['CV F1'], width=w, label='CV F1', color='#9b5de5')
ax.bar(x + w / 2, compare_cv_test['Test F1'], width=w, label='Test F1', color='#00bbf9')
ax.set_xticks(x)
ax.set_xticklabels(compare_cv_test['method'], rotation=15, ha='right')
ax.set_ylabel('F1')
ax.set_title('CV vs Test F1 - every method')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(22, 4.5))
for ax, (label, model), X_te in zip(
        axes,
        [('Baseline', fitted['Baseline']),
         ('GridSearchCV', fitted['Grid']),
         ('Optuna', fitted['Optuna']),
         ('PSO (pyswarm)', fitted['PSO'])],
        [X_test_full, X_test, X_test, X_test, X_test],
):
    y_pred = model.predict(X_te)
    cm = manual_confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Purples', ax=ax, cbar=False,
                xticklabels=['Not adopted', 'Adopted'], yticklabels=['Not adopted', 'Adopted'])
    ax.set_title(f'{label}\nF1={manual_f1(y_test, y_pred):.3f}')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')
plt.tight_layout()
plt.show()

In [ ]:
best_test_idx = test_results['f1'].idxmax()
best_label = test_results.loc[best_test_idx, 'method']
print(f"Best model on the test set: {best_label} (F1={test_results.loc[best_test_idx, 'f1']:.4f})")

if 'Grid' in best_label:
    BEST_MODEL = fitted['Grid']
    BEST_FEATS = final_features
elif 'Optuna' in best_label:
    BEST_MODEL = fitted['Optuna']
    BEST_FEATS = final_features
elif 'Genetic' in best_label:
    BEST_MODEL = fitted['GA']
    BEST_FEATS = final_features
elif 'PSO' in best_label:
    BEST_MODEL = fitted['PSO']
    BEST_FEATS = final_features
else:
    BEST_MODEL = fitted['Baseline']
    BEST_FEATS = feature_cols

print(f'Feature set used: {len(BEST_FEATS)} features')

## Model explainability

###  Built-in CatBoost feature importance

In [ ]:
importances = BEST_MODEL.get_feature_importance()
imp_df = pd.DataFrame({'feature': BEST_FEATS, 'importance': importances}).sort_values('importance', ascending=True)

fig, ax = plt.subplots(figsize=(9, max(4, 0.35 * len(imp_df))))
ax.barh(imp_df['feature'], imp_df['importance'], color='#9b5de5')
ax.set_xlabel('Importance (PredictionValuesChange)')
ax.set_title(f'CatBoost built-in feature importance - {best_label}')
plt.tight_layout()
plt.show()

### SHAP values

In [ ]:
explainer = shap.TreeExplainer(BEST_MODEL)
shap_values = explainer.shap_values(X_te)

if isinstance(shap_values, list):
    shap_values_pos = shap_values[1]
else:
    shap_values_pos = shap_values

print(f'SHAP values shape: {shap_values_pos.shape}')

In [ ]:
shap.summary_plot(
    shap_values_pos, X_te, feature_names=BEST_FEATS,
    plot_type='bar', show=False, color='#9b5de5'
)
plt.title('SHAP- mean |contribution| per feature')
plt.tight_layout()
plt.show()

In [ ]:
y_proba_te = BEST_MODEL.predict_proba(X_te)[:, 1]
idx_high = int(np.argmax(y_proba_te))
idx_low = int(np.argmin(y_proba_te))

print(f'Sample {idx_high}: true={y_test[idx_high]}, p(adopted)={y_proba_te[idx_high]:.3f}')
print(f'Sample {idx_low}:  true={y_test[idx_low]},  p(adopted)={y_proba_te[idx_low]:.3f}')

In [ ]:
fig = plt.figure(figsize=(14, 3))
shap.force_plot(
    explainer.expected_value if not isinstance(explainer.expected_value, (list, np.ndarray))
    else (explainer.expected_value[1] if isinstance(explainer.expected_value, list) else explainer.expected_value),
    shap_values_pos[idx_high],
    pd.Series(X_te[idx_high], index=BEST_FEATS),
    matplotlib=True, show=False
)
plt.title(f'Why was sample {idx_high} predicted ADOPTED (p={y_proba_te[idx_high]:.3f})?')
plt.tight_layout();
plt.show()

In [ ]:
fig = plt.figure(figsize=(14, 3))
shap.force_plot(
    explainer.expected_value if not isinstance(explainer.expected_value, (list, np.ndarray))
    else (explainer.expected_value[1] if isinstance(explainer.expected_value, list) else explainer.expected_value),
    shap_values_pos[idx_low],
    pd.Series(X_te[idx_low], index=BEST_FEATS),
    matplotlib=True, show=False
)
plt.title(f'Why was sample {idx_low} predicted NOT ADOPTED (p={y_proba_te[idx_low]:.3f})?')
plt.tight_layout()
plt.show()

In [ ]:
# Dependence plot for the most important feature
top_feature_idx = int(np.argmax(np.abs(shap_values_pos).mean(axis=0)))
top_feature = BEST_FEATS[top_feature_idx]
print(f'Top feature by mean |SHAP|: {top_feature}')

shap.dependence_plot(
    top_feature_idx, shap_values_pos, X_te,
    feature_names=BEST_FEATS, show=False
)
plt.title(f'SHAP dependence plot- {top_feature}')
plt.tight_layout()
plt.show()